In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os
import scanpy as sc
import plotnine as gg

import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.colors as mcolors
from tqdm import tqdm


tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
f_vector_cols = [
    # "N Mismatch_y",
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]


timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
ops_summarystats_df = (
    df_.pivot(index=["sgRNA", "Gene", "N Mismatch"], columns="Variable(s)", values="Value")
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)
ops_summarystats_df


def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["tsne_x"] = embeddings_tsne[:, 0]
df_embeddings["tsne_y"] = embeddings_tsne[:, 1]
df_embeddings.info()

pca_ = PCA(n_components=2)
pca_.fit(df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]])
embeddings_pca = pca_.transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["pca_x"] = embeddings_pca[:, 0]
df_embeddings["pca_y"] = embeddings_pca[:, 1]
# comparison with growth fitness screen

ops_df = df_embeddings.merge(ops_summarystats_df.reset_index(), on="sgRNA", how="left")
# ops_df

In [ ]:
ops_df_no_mismatch = ops_df.loc[lambda x: x["N Mismatch_y"] == 0]
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    ops_df_no_mismatch[[c for c in ops_df_no_mismatch.columns if c.startswith("Feature Vector")]]
)
ops_df_no_mismatch["tsne_x"] = embeddings_tsne[:, 0]
ops_df_no_mismatch["tsne_y"] = embeddings_tsne[:, 1]

In [ ]:
(
    gg.ggplot(
        ops_df_no_mismatch,
        gg.aes(x="tsne_x", y="tsne_y", color="Instantaneous Growth Rate: Volume"),
    )
    + gg.geom_point()
    + gg.labs(color="vol. growth")
    + gg.theme_minimal()
)

In [ ]:
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
transcript_obs = adata_case.obs.copy()
transcript_cluster_assignment = (
    transcript_obs.groupby("spacer")["annotated_leiden_case_coarse"]
    .apply(lambda x: x.mode()[0])
    .to_frame("transcript_cluster_assignment")
    .reset_index()
)
transcript_cluster_assignment

In [ ]:
import sys

sys.path.append("/workspace/experiments/12312025_surrogate")

from data_resources import load_fitness_data

fitness_df = load_fitness_data().reset_index()

In [ ]:
fitness_df

In [ ]:
ops_df_no_mismatch_ = ops_df_no_mismatch.merge(
    transcript_cluster_assignment, right_on="spacer", left_on="sgRNA", how="left"
).merge(fitness_df, left_on="sgRNA", right_on="spacer", how="left")
ops_df_no_mismatch_
ops_df_no_mismatch_

In [ ]:
(
    gg.ggplot(
        ops_df_no_mismatch_,
        gg.aes(x="tsne_x", y="tsne_y", color="transcript_cluster_assignment"),
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.labs(x="TSNE 1", y="TSNE 2", color="transcript cluster")
    + gg.theme(
        figure_size=(6, 3),
    )
)

In [ ]:
ops_df_select = ops_df_no_mismatch_.loc[lambda x: x["gene"].isin(["lpxA", "lpxB", "lpxD", "lpxK"])]
ops_df_select["gene"] = pd.Categorical(
    ops_df_select["gene"], categories=["lpxA", "lpxB", "lpxD", "lpxK"]
)
(
    gg.ggplot(
        ops_df_no_mismatch_,
        gg.aes(x="tsne_x", y="tsne_y"),
    )
    + gg.geom_point(size=0.5)
    + gg.geom_point(ops_df_select, gg.aes(x="tsne_x", y="tsne_y", color="gene"), size=3)
    + gg.theme_minimal()
    + gg.labs(x="TSNE 1", y="TSNE 2", color="transcript cluster")
    + gg.theme(
        figure_size=(6, 3),
    )
)

In [ ]:
(
    gg.ggplot(
        ops_df_no_mismatch_,
        gg.aes(x="tsne_x", y="tsne_y", color="transcript_cluster_assignment"),
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.labs(x="TSNE 1", y="TSNE 2", color="transcript cluster")
    + gg.theme(figure_size=(6, 3))
)

In [ ]:
(
    gg.ggplot(
        ops_df_no_mismatch_,
        gg.aes(x="tsne_x", y="tsne_y", color="T4"),
    )
    + gg.geom_point()
    + gg.scale_color_cmap(cmap_name="bwr", limits=(-2, 2))
    + gg.theme_minimal()
)